# Multitask training — GS40 + GS55 + GS33 combined

Train **one** ConvNeXt + StarDist multitask model on tiles from all three gestational ages.

## Prerequisites

For **each** cohort you must have already run the tile extraction pipeline:

- PNG tiles under `…/train/images` and `…/val/images`
- Instance masks + `*_inst2class.json` sidecars under `…/train/labels` and `…/val/labels`
- Split CSVs (`train.csv` / `val.csv`) with one tile stem per row

> **GS33 note:** run `make_training_dataset/GS33/2_build_dataset_gs33_256.ipynb` first.

## Quick start

1. Edit **section 2 — PARAMETERS** (paths, hyperparameters).
2. Run sections **3→4** (load stems, write `config_gs40_gs55_gs33_multitask.yaml`).
3. **Run training:** use **section 5** — terminal (recommended) or the notebook cell below.
4. **Fine-tune only:** set `RESUME_CHECKPOINT` in PARAMETERS, then re-run **2→5**.

## How this notebook works

1. **Parameters** — set paths for all three cohorts and hyperparameters.
2. **Load stems** — reads train/val stems from each cohort's split CSVs and **concatenates** them.
3. **Write config** — builds a YAML where `data.train_sources` / `data.val_sources` list three roots.
4. **Run** — launches `python -m shared_convnext_stardist_decoder.train_v2`.

---
## 1 · Imports

In [ ]:
from pathlib import Path

from train_utils import build_training_config, read_stems, run_training, write_config

print("OK — train_utils loaded")

---
## 2 · PARAMETERS  ← edit here

**Image / label dirs** must match what you used when building `inst2class` files.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# GS40 (gestational 40)
# ═══════════════════════════════════════════════════════════════════════════
GS40_ROOT = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\dataset_256_40k_48_slides")
GS40_TRAIN_IMAGES = GS40_ROOT / "train" / "images"
GS40_INST_LABELS  = GS40_ROOT / "stardist_multitask_ready" / "train_instance_labels"
# GS40 val tiles live in the same image folder as train; only stems differ
GS40_VAL_IMAGES   = GS40_TRAIN_IMAGES
GS40_VAL_LABELS   = GS40_INST_LABELS
GS40_SPLIT_TRAIN  = GS40_ROOT / "splits" / "fold_0" / "train.csv"
GS40_SPLIT_VAL    = GS40_ROOT / "splits" / "fold_0" / "val.csv"

# ═══════════════════════════════════════════════════════════════════════════
# GS55 — 50k tiles, class-balanced sampling
# ═══════════════════════════════════════════════════════════════════════════
GS55_ROOT = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS55\cellvit_training\dataset_256_50k_clsbal_GS55")
GS55_TRAIN_IMAGES = GS55_ROOT / "train" / "images"
GS55_INST_LABELS  = GS55_ROOT / "train" / "labels"
GS55_VAL_IMAGES   = GS55_ROOT / "val" / "images"
GS55_VAL_LABELS   = GS55_ROOT / "val" / "labels"
GS55_SPLIT_TRAIN  = GS55_ROOT / "splits" / "fold_0" / "train.csv"
GS55_SPLIT_VAL    = GS55_ROOT / "splits" / "fold_0" / "val.csv"

# ═══════════════════════════════════════════════════════════════════════════
# GS33 — 30k tiles, class-balanced sampling (1-in-10 slide subset)
# ═══════════════════════════════════════════════════════════════════════════
GS33_ROOT = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS33\cellvit_training\dataset_256_30k_GS33")
GS33_TRAIN_IMAGES = GS33_ROOT / "train" / "images"
GS33_INST_LABELS  = GS33_ROOT / "train" / "labels"
GS33_VAL_IMAGES   = GS33_ROOT / "val" / "images"
GS33_VAL_LABELS   = GS33_ROOT / "val" / "labels"
GS33_SPLIT_TRAIN  = GS33_ROOT / "splits" / "fold_0" / "train.csv"
GS33_SPLIT_VAL    = GS33_ROOT / "splits" / "fold_0" / "val.csv"

# ═══════════════════════════════════════════════════════════════════════════
# Output & repo
# ═══════════════════════════════════════════════════════════════════════════
REPO_ROOT   = Path(r"C:\Users\Andre\cursor_projects\Convnext_stardist")
CONFIG_PATH = REPO_ROOT / "shared_convnext_stardist_decoder" / "config_gs40_gs55_gs33_multitask.yaml"
CKPT_OUT    = GS40_ROOT / "convnext_stardist_multitask_runs" / "run_gs40_gs55_gs33_v4"

EXPERIMENT_NAME = "convnext_stardist_mt_gs40_gs55_gs33_v4"

# Class names — same order as GS40 training (lowercase "gi" for model compatibility)
CLASS_NAMES = [
    "bone", "brain", "eye", "heart", "lungs", "gi", "liver", "spleen",
    "pancreas", "kidney", "mesokidney", "collagen", "ear", "nontissue",
    "thymus", "thyroid", "bladder", "skull", "spleen2",
]

# Training hyperparameters
EPOCHS                 = 35
BATCH_SIZE             = 8
LR                     = 5e-5
LR_MIN                 = 1e-6
WEIGHT_DECAY           = 0.03
FREEZE_BACKBONE_EPOCHS = 10
LOSS_W_PROB            = 2.0
LOSS_W_DIST            = 0.15
LOSS_W_CLS             = 1.0
LOSS_W_INST            = 0.25
CLS_SEMANTIC_DIM       = 128
NUM_WORKERS            = 8

# Fine-tuning: set to a .pt path to resume, else None
RESUME_CHECKPOINT = None
RESUME_STRICT     = True

CKPT_OUT.mkdir(parents=True, exist_ok=True)
print(f"Experiment : {EXPERIMENT_NAME}")
print(f"Config     → {CONFIG_PATH}")
print(f"Checkpoints → {CKPT_OUT}")
print(f"\nGS40 train images : {GS40_TRAIN_IMAGES}")
print(f"GS55 train images : {GS55_TRAIN_IMAGES}")
print(f"GS33 train images : {GS33_TRAIN_IMAGES}")

---
## 3 · Load and merge train / val stems

In [ ]:
def _load_split(csv_path: Path, label: str) -> list[str]:
    if not csv_path.is_file():
        raise FileNotFoundError(f"Missing {label}: {csv_path}")
    stems = read_stems(csv_path)
    print(f"{label}: {len(stems):,} stems from {csv_path.name}")
    return stems

train_40 = _load_split(GS40_SPLIT_TRAIN, "GS40 train")
val_40   = _load_split(GS40_SPLIT_VAL,   "GS40 val")
train_55 = _load_split(GS55_SPLIT_TRAIN, "GS55 train")
val_55   = _load_split(GS55_SPLIT_VAL,   "GS55 val")
train_33 = _load_split(GS33_SPLIT_TRAIN, "GS33 train")
val_33   = _load_split(GS33_SPLIT_VAL,   "GS33 val")

train_stems = train_40 + train_55 + train_33
val_stems   = val_40   + val_55   + val_33

def _dedupe(xs: list[str]) -> list[str]:
    seen: set[str] = set()
    out: list[str] = []
    for s in xs:
        if s not in seen:
            seen.add(s)
            out.append(s)
    return out

train_stems = _dedupe(train_stems)
val_stems   = _dedupe(val_stems)

overlap = set(train_stems) & set(val_stems)
print(f"\nCombined : train {len(train_stems):,}  |  val {len(val_stems):,}")
print(f"  GS40   : {len(train_40):,} / {len(val_40):,}")
print(f"  GS55   : {len(train_55):,} / {len(val_55):,}")
print(f"  GS33   : {len(train_33):,} / {len(val_33):,}")
print(f"Train∩Val stem overlap: {len(overlap)} tiles (should be 0)")

---
## 4 · Build config and write YAML

`build_training_config` sets model/train/checkpoint defaults. The **`data`** section is replaced with `train_sources` + `val_sources` so all three cohorts are loaded.

In [ ]:
cfg = build_training_config(
    experiment_name=EXPERIMENT_NAME,
    train_images_dir=GS40_TRAIN_IMAGES,
    inst_labels_dir=GS40_INST_LABELS,
    ckpt_out=CKPT_OUT,
    train_stems=train_stems,
    val_stems=val_stems,
    class_names=CLASS_NAMES,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    lr_min=LR_MIN,
    weight_decay=WEIGHT_DECAY,
    freeze_backbone_epochs=FREEZE_BACKBONE_EPOCHS,
    loss_w_cls=LOSS_W_CLS,
    loss_w_inst=LOSS_W_INST,
    loss_w_dist=LOSS_W_DIST,
    cls_semantic_dim=CLS_SEMANTIC_DIM,
)

cfg["data"] = {
    "train_stems": train_stems,
    "val_stems":   val_stems,
    "train_sources": [
        {"images_dir": str(GS40_TRAIN_IMAGES), "labels_dir": str(GS40_INST_LABELS)},
        {"images_dir": str(GS55_TRAIN_IMAGES), "labels_dir": str(GS55_INST_LABELS)},
        {"images_dir": str(GS33_TRAIN_IMAGES), "labels_dir": str(GS33_INST_LABELS)},
    ],
    "val_sources": [
        {"images_dir": str(GS40_VAL_IMAGES),   "labels_dir": str(GS40_VAL_LABELS)},
        {"images_dir": str(GS55_VAL_IMAGES),   "labels_dir": str(GS55_VAL_LABELS)},
        {"images_dir": str(GS33_VAL_IMAGES),   "labels_dir": str(GS33_VAL_LABELS)},
    ],
    "cache_to_ram": False,
}

cfg["train"]["loss_w_prob"] = LOSS_W_PROB
cfg["train"]["num_workers"] = NUM_WORKERS

write_config(cfg, CONFIG_PATH)
print("\nMulti-source data section:")
print("  train_sources:", len(cfg["data"]["train_sources"]), "roots")
print("  val_sources:  ", len(cfg["data"]["val_sources"]),   "roots")
print(f"\nConfig written → {CONFIG_PATH}")

---
## 4b · Write finetune config (same splits, new experiment)

Writes `config_finetune_gs40_gs55_gs33.yaml` using **identical** `train_stems` / `val_stems` from section 3.

In [ ]:
FINETUNE_CONFIG_PATH = REPO_ROOT / "shared_convnext_stardist_decoder" / "config_finetune_gs40_gs55_gs33.yaml"
FINETUNE_CKPT_OUT    = GS40_ROOT / "convnext_stardist_multitask_runs" / "run_gs40_gs55_gs33_v4_finetune"
FINETUNE_EXPERIMENT  = "convnext_stardist_mt_gs40_gs55_gs33_v4_finetune"

ft_cfg = build_training_config(
    experiment_name=FINETUNE_EXPERIMENT,
    train_images_dir=GS40_TRAIN_IMAGES,
    inst_labels_dir=GS40_INST_LABELS,
    ckpt_out=FINETUNE_CKPT_OUT,
    train_stems=train_stems,
    val_stems=val_stems,
    class_names=CLASS_NAMES,
    epochs=15,
    batch_size=BATCH_SIZE,
    lr=1e-5,
    lr_min=1e-7,
    weight_decay=WEIGHT_DECAY,
    freeze_backbone_epochs=5,
    loss_w_cls=LOSS_W_CLS,
    loss_w_inst=LOSS_W_INST,
    loss_w_dist=LOSS_W_DIST,
    cls_semantic_dim=CLS_SEMANTIC_DIM,
)

ft_cfg["data"] = {
    "train_stems": train_stems,
    "val_stems":   val_stems,
    "train_sources": [
        {"images_dir": str(GS40_TRAIN_IMAGES), "labels_dir": str(GS40_INST_LABELS)},
        {"images_dir": str(GS55_TRAIN_IMAGES), "labels_dir": str(GS55_INST_LABELS)},
        {"images_dir": str(GS33_TRAIN_IMAGES), "labels_dir": str(GS33_INST_LABELS)},
    ],
    "val_sources": [
        {"images_dir": str(GS40_VAL_IMAGES),   "labels_dir": str(GS40_VAL_LABELS)},
        {"images_dir": str(GS55_VAL_IMAGES),   "labels_dir": str(GS55_VAL_LABELS)},
        {"images_dir": str(GS33_VAL_IMAGES),   "labels_dir": str(GS33_VAL_LABELS)},
    ],
    "cache_to_ram": False,
}

ft_cfg["train"]["loss_w_prob"] = LOSS_W_PROB
ft_cfg["train"]["num_workers"] = NUM_WORKERS

write_config(ft_cfg, FINETUNE_CONFIG_PATH)
print("Finetune config written with same splits as main run.")
print(f"Config → {FINETUNE_CONFIG_PATH}")

---
## 5 · Run training

**From terminal (recommended — live progress visible):**

```powershell
conda activate convnext_stardist_mt
cd C:\Users\Andre\cursor_projects\Convnext_stardist

# Fresh run (GS40 + GS55 + GS33)
python -m shared_convnext_stardist_decoder.train_v2 --config shared_convnext_stardist_decoder\config_gs40_gs55_gs33_multitask.yaml

# Fine-tune from best checkpoint
python -m shared_convnext_stardist_decoder.train_v2 ^
    --config shared_convnext_stardist_decoder\config_finetune_gs40_gs55_gs33.yaml ^
    --resume <path\to\best.pt>
```

**What to watch in the log:**

| Column | Good sign |
|--------|-----------|
| `bce` | Decreases epoch 1→50 |
| `cls_px` | Decreases from ~1.5 toward <0.8 |
| `val cls_px` | Tracks train; large gap = overfitting → use `best.pt` |
| `dist` | Decreases toward <1.0 |

In [ ]:
# Run inside notebook — or use the terminal command in section 5 instead.
# Set RESUME_CHECKPOINT and RESUME_STRICT in PARAMETERS (section 2) to fine-tune.

# exit_code = run_training(
#     config_path       = CONFIG_PATH,
#     repo_root         = REPO_ROOT,
#     resume_checkpoint = RESUME_CHECKPOINT,
#     resume_strict     = RESUME_STRICT,
# )
# assert exit_code == 0, f"Training failed with exit code {exit_code}"